In [2]:
import os, numpy as np, pandas as pd
from scipy.stats import spearmanr, rankdata
from pybiomart import Server

t = os.path.expanduser("~/FYP_Regeneration/results/tables/")
ax = pd.read_csv(t+"axolotl_regen_vs_uninjured_all.csv")
ms = pd.read_csv(t+"mouse_sc_fibroblast_regen_vs_uninjured_all.csv")
hu = pd.read_csv(t+"human_fibroblast_wound_vs_skin_all.csv")
for d in (ax, ms, hu): d["g"] = d["gene"].astype(str).str.upper()

srv = Server(host="http://www.ensembl.org")
ds = srv.marts["ENSEMBL_MART_ENSEMBL"].datasets["mmusculus_gene_ensembl"]
hom = ds.query(attributes=["external_gene_name","hsapiens_homolog_associated_gene_name"]).dropna()
m2h = dict(zip(hom.iloc[:,0].str.upper(), hom.iloc[:,1].str.upper()))
ms_h = ms.copy(); ms_h["g"] = ms_h["g"].map(m2h); ms_h = ms_h.dropna(subset=["g"])

def pair(a, b, cap, n_perm=10000, seed=42):
    m = a[["g","logfoldchanges"]].merge(b[["g","logfoldchanges"]], on="g", suffixes=("_a","_b"))
    m = m.replace([np.inf,-np.inf], np.nan).dropna()
    if cap: m = m[(m["logfoldchanges_a"].abs()<=cap) & (m["logfoldchanges_b"].abs()<=cap)]
    x, y = m["logfoldchanges_a"].values, m["logfoldchanges_b"].values
    rho = spearmanr(x, y)[0]
    rx, ry = rankdata(x), rankdata(y)
    rng = np.random.default_rng(seed)
    null = np.array([np.corrcoef(rx, rng.permutation(ry))[0,1] for _ in range(n_perm)])
    p = (np.sum(np.abs(null) >= abs(rho)) + 1) / (n_perm + 1)
    return len(m), rho, p, (np.sign(x)==np.sign(y)).mean()

rows = []
for cap in [None, 10, 5]:
    for name, a, b in [("Axolotl-Mouse", ax, ms), ("Axolotl-Human", ax, hu), ("Mouse-Human", ms_h, hu)]:
        n, rho, p, conc = pair(a, b, cap)
        rows.append({"cap": cap or "none", "pair": name, "n": n, "rho": round(rho,4),
                     "perm_p": round(p,5), "same_direction": round(conc,3)})
res = pd.DataFrame(rows)
print(res.to_string(index=False))
res.to_csv(t+"correlation_fold_change_cap_robustness.csv", index=False)

 cap          pair     n    rho  perm_p  same_direction
none Axolotl-Mouse 11544 0.0845  0.0001           0.622
none Axolotl-Human 11777 0.1181  0.0001           0.548
none   Mouse-Human 16956 0.1410  0.0001           0.530
  10 Axolotl-Mouse  9817 0.1621  0.0001           0.632
  10 Axolotl-Human 10012 0.1785  0.0001           0.560
  10   Mouse-Human 14818 0.2276  0.0001           0.554
   5 Axolotl-Mouse  9760 0.1614  0.0001           0.631
   5 Axolotl-Human  9941 0.1788  0.0001           0.559
   5   Mouse-Human 14751 0.2277  0.0001           0.555


In [3]:
s_genes = ["MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG","GINS2","MCM6","CDCA7","DTL","PRIM1","UHRF1","MLF1IP","HELLS","RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76","SLBP","CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51","RRM2","CDC45","CDC6","EXO1","TIPIN","DSCC1","BLM","CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8"]
g2m_genes = ["HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A","NDC80","CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF","TACC3","FAM64A","SMC4","CCNB2","CKAP2L","CKAP2","AURKB","BUB1","KIF11","ANP32E","TUBB4B","GTSE1","KIF20B","HJURP","CDCA3","HN1","CDC20","TTK","CDC25C","KIF2C","RANGAP1","NCAPD2","DLGAP5","CDCA2","CDCA8","ECT2","KIF23","HMMR","AURKA","PSRC1","ANLN","LBR","CKAP5","CENPE","CTCF","NEK2","G2E3","GAS2L3","CBX5","CENPA"]
cc = set(s_genes) | set(g2m_genes)

rows = []
for cap in [None, 10]:
    for name, a, b in [("Axolotl-Mouse", ax, ms), ("Axolotl-Human", ax, hu), ("Mouse-Human", ms_h, hu)]:
        n, rho, p, conc = pair(a[~a["g"].isin(cc)], b[~b["g"].isin(cc)], cap)
        rows.append({"cap": cap or "none", "pair": name, "n": n, "rho": round(rho,4), "perm_p": round(p,5)})
res_cc = pd.DataFrame(rows)
print(res_cc.to_string(index=False))
res_cc.to_csv(t+"correlation_no_cell_cycle_robustness.csv", index=False)

 cap          pair     n    rho  perm_p
none Axolotl-Mouse 11454 0.0772  0.0001
none Axolotl-Human 11687 0.1115  0.0001
none   Mouse-Human 16865 0.1364  0.0001
  10 Axolotl-Mouse  9732 0.1519  0.0001
  10 Axolotl-Human  9927 0.1686  0.0001
  10   Mouse-Human 14727 0.2209  0.0001
